In [2]:
import pandas as pd
import os
from getpass import getpass
from openai import OpenAI

# 1. Load the dataset
df = pd.read_csv("dataset_clean.csv")

# Display basic info about the dataframe
print(f"Columns: {df.columns}")
print(f"Total rows: {len(df)}")
df.head()

# 2. Setup OpenAI Client
# Securely enter the API key
os.environ["OPENAI_API_KEY"] = getpass("Enter your API key: ")
client = OpenAI()

def answer_question(question):
    """
    Sends a tax-related question to the OpenAI API and returns a concise answer.
    The prompt forces the model to stick to Austrian tax law with no formatting.
    """
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": f"""Beantworte die folgende steuerrechtliche Frage für Österreich kurz und präzise.
Gib nur die Antwort, keine Einleitung. Reiner Text: kein Fettgeschriebenes etc..
Wenn möglich, nenne die relevante gesetzliche Grundlage.
Question:
{question}"""
            }
        ]
    )
    return response.choices[0].message.content.strip()

def process_dataframe(input_df, output_filename):
    """
    Iterates through a dataframe, fetches answers, and saves them to a CSV.
    """
    results = []

    for i, row in input_df.iterrows():
        qid = row["id"]
        question = row["prompt"]

        try:
            answer = answer_question(question)
        except Exception as e:
            print(f"Error at ID {qid}: {e}")
            answer = "ERROR"

        results.append({
            "id": qid,
            "answer": answer
        })

        # Print progress every 20 rows
        if (i + 1) % 20 == 0:
            print(f"Processed {i+1}/{len(input_df)} rows...")

    # Save results to CSV
    output_df = pd.DataFrame(results)
    output_df.to_csv(output_filename, index=False)
    print(f"Finished! Saved to {output_filename}")
    return output_df

# 3. Execution: Small Sample Test (10 rows)
print("Starting sample test...")
sample_df = df.sample(10, random_state=42).reset_index(drop=True)
sample_output = process_dataframe(sample_df, "sample_output.csv")
print(sample_output.head())

# 4. Execution: Full Dataset
print("Starting full inference...")
final_output = process_dataframe(df, "inference_output.csv")

Columns: Index(['id', 'prompt'], dtype='object')
Total rows: 643
Enter your API key: ··········
Starting sample test...
Finished! Saved to sample_output.csv
             id                                             answer
0      SELF-070  Eine Sicherheitseinrichtung für Registrierkass...
1  VAT-INTL-047  Bei einer Leistung an ein Unternehmen gilt der...
2       EMP-045  Nein. Wenn das Klimaticket anstelle einer Geha...
3  GRESt-AT-051  Die Selbstberechnung der Grunderwerbsteuer dar...
4  TAX-INTL-013  Ja. Für die österreichische Besteuerung ist gr...
Starting full inference...
Processed 20/643 rows...
Processed 40/643 rows...
Processed 60/643 rows...
Processed 80/643 rows...
Processed 100/643 rows...
Processed 120/643 rows...
Processed 140/643 rows...
Processed 160/643 rows...
Processed 180/643 rows...
Processed 200/643 rows...
Processed 220/643 rows...
Processed 240/643 rows...
Processed 260/643 rows...
Processed 280/643 rows...
Processed 300/643 rows...
Processed 320/643 rows...
Pr